In [1]:
%pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install transformers datasets

: 

: 

: 

In [1]:
from datasets import load_dataset
# dataset = load_dataset("csv", data_files="IMDB Dataset.csv")
dataset= load_dataset("imdb")

/home/harpreet/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

: 

: 

: 

In [2]:
dataset["train"][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [12]:
print("hello")

hello


Pre process the data

In [3]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

Tokenize dataset

In [4]:
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


: 

: 

: 

In [5]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1818.85it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

: 

: 

: 

In [6]:
tokenized_dataset["train"][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
%pip install transformers[torch]

Note: you may need to restart the kernel to use updated packages.


: 

: 

: 

In [ ]:
%pip install -U accelerate

Note: you may need to restart the kernel to use updated packages.


: 

: 

: 

In [ ]:
import accelerate
print(accelerate.__version__)

1.13.0


: 

: 

: 

In [22]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch"
)

In [ ]:
# small_train = tokenized_dataset["train"].select(range(5))
# small_test = tokenized_dataset["test"].select(range(2))

In [23]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
)

In [24]:
trainer.train()

/home/harpreet/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,0.357024


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.87it/s]
/home/harpreet/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=1, training_loss=0.41453808546066284, metrics={'train_runtime': 21.7083, 'train_samples_per_second': 0.23, 'train_steps_per_second': 0.046, 'total_flos': 662336993280.0, 'train_loss': 0.41453808546066284, 'epoch': 1.0})

In [25]:
trainer.evaluate()

RuntimeError: on_train_begin must be called before on_evaluate

## Arxiv

In [26]:
%pip install arxiv

  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'sgmllib3k' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'sgmllib3k'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6089 sha256=233f0f948b8d211455b9f07e7006add1d06bcbcb501058c011cda350c9d6af0b
  Stored in directory: /home/harpreet/.cache/pip/wheels/3d/4d/ef/37cdccc18d6fd7e0dd7817dcdf9146d4d6789c32a227a28134
Successfully built sgmllib3k
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [arxiv]32m2/3 [arxiv]
Note: you may need to restart the kernel to use updated packages.


In [27]:
import arxiv
import pandas as pd

In [28]:
#query to 
query="ai or artificial intelligence or machine learning"
search=arxiv.Search(query=query,max_results=10,sort_by=arxiv.SortCriterion.SubmittedDate)

In [ ]:
client = arxiv.Client(
    page_size=5,
    delay_seconds=3
)

# search = arxiv.Search(
#     query="large language model",
#     max_results=3
# )
# Fetch results
# papers = []
papers = [
    {
        "title": "Test AI Paper",
        "abstract": "This is test abstract",
        "published": "2024"
    }
]

# for result in client.results(search):
#     papers.append({
#         "title": result.title,
#         # "Authors": ", ".join([author.name for author in result.authors]),
#         # "published": result.published,
#         # "abstract": result.summary,
#         # "categories": result.categories
#     })
df= pd.DataFrame(papers)
    


In [37]:
df = pd.DataFrame(papers)
abst= df['abstract'][0]
print(df['abstract'])


0    This is test abstract
Name: abstract, dtype: str


In [ ]:
from transformers import pipeline

summarizer = pipeline(
    "text-generation",
    model="google/flan-t5-small"
)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2767.77it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTe

NameError: name 'text' is not defined

In [ ]:
summary = summarizer(
    "summarize: Artificial intelligence is transforming industries...",
    max_length=50
)
print(summary)

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'summarize: Artificial intelligence is transforming industries...transforming the way they conduct, and the way they do what they do.'}]
